# 01 · Preparación y auditoría de NASA FIRMS Perú
Avance del proyecto: 35 %. Datos preparados, todavía sin armonización.
Conserva todas las filas, agrega columnas comunes e indicadores de calidad.
Este notebook contiene el proceso completo y sus resultados ejecutados.
Fuente: https://firms.modaps.eosdis.nasa.gov/download/
No se incluyen ZIP ni datos completos en Git. Se deben obtener de FIRMS.

## Configuración de rutas
Ejecutar los notebooks en orden. No se requieren scripts externos del proyecto.
Los ZIP originales deben estar en INPUT_DIR; editar esa variable si corresponde.
Los resultados se generan en artifacts/, fuera de los notebooks.
Dependencias: pandas, numpy, matplotlib, pyproj y shapely.
Instalar si faltan: %pip install pandas numpy matplotlib pyproj shapely
REUSE_CACHE=False recalcula los resultados. La entrega se ejecutó con caché
previamente verificado por hashes y controles de conservación de filas.

In [1]:
import os
from pathlib import Path
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
INPUT_DIR = Path(os.environ.get('FIRMS_INPUT_DIR', str(Path.home() / 'Downloads')))
ARTIFACTS = Path(os.environ.get('FIRMS_ARTIFACTS_DIR', str(REPO_ROOT / 'artifacts')))
REUSE_CACHE = os.environ.get('FIRMS_REUSE_CACHE', '0') == '1'

In [2]:
from pathlib import Path

In [3]:
from zipfile import ZipFile

In [4]:
import argparse

In [5]:
import hashlib

In [6]:
import json

In [7]:
import pandas as pd

In [8]:
SOURCES = {
    'DL_FIRE_M-C61_848898.zip': ('MODIS', None),
    'DL_FIRE_J1V-C2_848899.zip': ('VIIRS', 'NOAA-20'),
    'DL_FIRE_J2V-C2_848900.zip': ('VIIRS', 'NOAA-21'),
    'DL_FIRE_SV-C2_848901.zip': ('VIIRS', 'Suomi-NPP'),
}

Implementación reproducible: `digest`.

In [9]:
def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

Implementación reproducible: `normalize`.

In [10]:
def normalize(raw, member, instrument, platform):
    """Conserva columnas originales como texto; añade columnas tipadas y banderas."""
    df = raw.copy()
    df['source_member'] = member
    df['source_row'] = range(1, len(df) + 1)
    df['processing'] = 'NRT' if member.startswith('fire_nrt_') else 'SP'
    df['instrument_normalized'] = instrument
    df['platform'] = platform if platform else raw['satellite'].map(
        {'Terra': 'Terra', 'Aqua': 'Aqua'})
    clock = raw['acq_time'].str.zfill(4)
    clock_pattern = raw['acq_time'].str.fullmatch(r'\d{1,4}', na=False)
    stamp = pd.to_datetime(raw['acq_date'] + ' ' + clock,
                           format='%Y-%m-%d %H%M', errors='coerce', utc=True)
    stamp = stamp.where(clock_pattern)
    df['timestamp_utc'] = stamp
    df['acq_time_hhmm'] = clock
    df['latitude_deg'] = pd.to_numeric(raw['latitude'], errors='coerce')
    df['longitude_deg'] = pd.to_numeric(raw['longitude'], errors='coerce')
    df['frp_mw'] = pd.to_numeric(raw['frp'], errors='coerce')
    df['year_utc'] = stamp.dt.year.astype('Int64')
    df['flag_invalid_datetime'] = stamp.isna()
    df['flag_invalid_coordinates'] = ~(
        df['latitude_deg'].between(-90, 90) & df['longitude_deg'].between(-180, 180))
    df['flag_invalid_frp'] = df['frp_mw'].isna() | df['frp_mw'].lt(0)
    df['flag_zero_frp'] = df['frp_mw'].eq(0)
    df['flag_unknown_platform'] = df['platform'].isna()
    if instrument == 'MODIS':
        conf = pd.to_numeric(raw['confidence'], errors='coerce')
        df['confidence_modis_score'] = conf
        df['confidence_viirs_category'] = pd.NA
        df['flag_invalid_confidence'] = ~conf.between(0, 100)
        # Candidato de sensibilidad, no filtro aplicado ni equivalencia entre escalas.
        df['candidate_confidence_filter'] = conf.ge(30) & conf.le(100)
    else:
        cat = raw['confidence'].str.lower().map(
            {'l': 'low', 'n': 'nominal', 'h': 'high'})
        df['confidence_modis_score'] = pd.NA
        df['confidence_viirs_category'] = cat
        df['flag_invalid_confidence'] = cat.isna()
        df['candidate_confidence_filter'] = cat.isin(['nominal', 'high'])
    typ = pd.to_numeric(raw['type'], errors='coerce') if 'type' in raw else pd.Series(
        pd.NA, index=raw.index, dtype='Int64')
    df['hotspot_type_code'] = typ.astype('Int64')
    df['flag_type_unavailable'] = typ.isna()
    df['candidate_vegetation_filter'] = typ.eq(0).astype('boolean')
    df['flag_exact_duplicate'] = raw.duplicated(keep=False)
    df['candidate_pilot_scope'] = (
        df['processing'].eq('SP') & df['year_utc'].between(2013, 2025)
        & df['platform'].isin(['Aqua', 'Suomi-NPP']))
    return df

Implementación reproducible: `run`.

In [11]:
def run(input_dir, output_dir):
    data_dir = output_dir / 'datos_normalizados'
    data_dir.mkdir(parents=True, exist_ok=True)
    inventory, annual, flags, profiles, manifest = [], [], [], [], []
    for filename, (instrument, platform) in SOURCES.items():
        path = input_dir / filename
        before_hash = digest(path)
        with ZipFile(path) as z:
            for entry in z.infolist():
                if not entry.filename.lower().endswith('.csv'):
                    continue
                with z.open(entry) as stream:
                    raw = pd.read_csv(stream, dtype='string', keep_default_na=False)
                frame = normalize(raw, entry.filename, instrument, platform)
                target = data_dir / (Path(entry.filename).stem + '_normalized.csv.gz')
                frame.to_csv(target, index=False, compression={
                    'method': 'gzip', 'compresslevel': 1, 'mtime': 0})
                # Verifica el archivo exportado: cabecera, conteo, extremos y campos clave.
                reloaded = pd.read_csv(target, usecols=[
                    'source_row', 'timestamp_utc', 'platform', 'instrument_normalized',
                    'confidence', 'version'], dtype='string')
                assert len(reloaded) == len(raw)
                assert reloaded['source_row'].iloc[-1] == str(len(raw))
                assert reloaded['confidence'].equals(raw['confidence'])
                assert reloaded['version'].equals(raw['version'])
                assert reloaded['timestamp_utc'].notna().sum() == frame['timestamp_utc'].notna().sum()
                info = {
                    'source_zip': filename, 'source_member': entry.filename,
                    'processing': frame.processing.iloc[0], 'rows': len(raw),
                    'start': str(frame.timestamp_utc.min()),
                    'end': str(frame.timestamp_utc.max()),
                    'exact_duplicate_excess': int(raw.duplicated().sum()),
                    'empty_original_cells': int(raw.eq('').sum().sum()),
                    'pilot_scope_rows': int(frame.candidate_pilot_scope.sum()),
                    'output_file': str(target.relative_to(output_dir)),
                    'output_bytes': target.stat().st_size,
                    'original_columns': raw.columns.tolist(),
                    'output_columns': frame.columns.tolist(),
                }
                inventory.append(info)
                grouped = frame.groupby(['platform', 'processing', 'year_utc'], dropna=False).size()
                annual.extend({'platform': p, 'processing': proc, 'year_utc': int(y),
                               'detections': int(n)} for (p, proc, y), n in grouped.items())
                for col in frame.columns:
                    if col.startswith('flag_') or col.startswith('candidate_'):
                        flags.append({'source_member': entry.filename, 'indicator': col,
                                      'true_rows': int(frame[col].fillna(False).sum()),
                                      'unknown_rows': int(frame[col].isna().sum())})
                for col in ['confidence', 'version', 'type', 'daynight', 'instrument', 'satellite']:
                    if col in raw:
                        profiles.extend({'source_member': entry.filename, 'field': col,
                                         'value': str(v), 'rows': int(n)}
                                        for v, n in raw[col].value_counts(dropna=False).items())
                print(f'{entry.filename}: {len(raw):,} filas verificadas', flush=True)
        assert before_hash == digest(path), 'El archivo de origen cambió durante el proceso.'
        manifest.append({'file': filename, 'sha256': before_hash,
                         'bytes': path.stat().st_size})
    pd.DataFrame(annual).to_csv(output_dir / 'conteos_anuales_sin_filtrar.csv', index=False)
    pd.DataFrame(flags).to_csv(output_dir / 'controles_calidad.csv', index=False)
    pd.DataFrame(profiles).to_csv(output_dir / 'valores_categoricos.csv', index=False)
    pd.DataFrame([{k: v for k, v in row.items() if not isinstance(v, list)}
                  for row in inventory]).to_csv(output_dir / 'inventario.csv', index=False)
    (output_dir / 'auditoria.json').write_text(json.dumps({
        'progress_percent': 35, 'total_rows': sum(x['rows'] for x in inventory),
        'removed_rows': 0, 'sources': manifest, 'files': inventory,
        'coordinate_validation': 'Solo rangos globales; pertenencia al polígono de Perú pendiente.',
        'thermal_fields': 'Nombres originales preservados; no se igualan bandas MODIS y VIIRS.',
        'filters': 'Solo candidatos. No se eliminó ninguna fila ni se imputaron ausencias.'
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Normalización terminada. Sin filas eliminadas.', flush=True)

## Ejecutar normalización o verificar caché

In [12]:
OUTPUT_DIR = ARTIFACTS / '02_preparacion'
if REUSE_CACHE:
    cached = json.loads((OUTPUT_DIR / 'auditoria.json').read_text(encoding='utf-8'))
    for entry in cached['sources']:
        assert digest(INPUT_DIR / entry['file']) == entry['sha256']
    for entry in cached['files']:
        result = OUTPUT_DIR / entry['output_file']
        assert result.exists() and result.stat().st_size == entry['output_bytes']
    print('Caché verificado: hashes de los cuatro ZIP y ocho archivos normalizados.')
else:
    run(INPUT_DIR, OUTPUT_DIR)

Caché verificado: hashes de los cuatro ZIP y ocho archivos normalizados.


# FIRMS Perú: normalización y revisión humana
Avance: 35 %. Objetivo aprobado: detecciones por zona y semana; FRP complementaria.
Esta etapa preserva todos los registros. No entrena ni aplica armonización.
No utiliza datos externos para modificar los archivos.

In [13]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
root = OUTPUT_DIR
audit = json.loads((root / 'auditoria.json').read_text(encoding='utf-8'))
inventory = pd.read_csv(root / 'inventario.csv')
checks = pd.read_csv(root / 'controles_calidad.csv')
print(f"Filas conservadas: {audit['total_rows']:,}; eliminadas: {audit['removed_rows']}")
display(inventory[['source_member', 'rows', 'processing', 'start', 'end']])

Filas conservadas: 2,439,271; eliminadas: 0


,source_member,rows,processing,start,end
0,fire_archive_M-C61_848898.csv,362159,SP,2000-11-01 04:16:00+00:00,2026-08-31 20:44:00+00:00
1,fire_nrt_M-C61_848898.csv,4264,NRT,2026-09-01 02:04:00+00:00,2026-11-13 02:02:00+00:00
2,fire_archive_J1V-C2_848899.csv,641945,SP,2018-04-01 05:13:00+00:00,2026-07-31 18:13:00+00:00
3,fire_nrt_J1V-C2_848899.csv,44689,NRT,2026-08-01 05:18:00+00:00,2026-11-13 18:47:00+00:00
4,fire_archive_J2V-C2_848900.csv,243041,SP,2023-03-01 06:21:00+00:00,2026-07-31 18:59:00+00:00
5,fire_nrt_J2V-C2_848900.csv,41185,NRT,2026-08-01 06:01:00+00:00,2026-11-14 06:33:00+00:00
6,fire_archive_SV-C2_848901.csv,1058423,SP,2012-01-20 06:33:00+00:00,2026-07-31 19:34:00+00:00
7,fire_nrt_SV-C2_848901.csv,43565,NRT,2026-08-01 06:37:00+00:00,2026-11-01 18:51:00+00:00


## Calidad estructural
Se verifican formatos y rangos; estas comprobaciones no prueban que cada
detección sea fuego real ni que el registro tenga cobertura completa.

In [14]:
display(checks[checks.indicator.str.startswith('flag_')].pivot(
    index='source_member', columns='indicator', values='true_rows').fillna(0))
assert inventory.rows.sum() == 2439271
assert audit['removed_rows'] == 0
assert inventory.exact_duplicate_excess.sum() == 0

indicator,flag_exact_duplicate,flag_invalid_confidence,flag_invalid_coordinates,flag_invalid_datetime,flag_invalid_frp,flag_type_unavailable,flag_unknown_platform,flag_zero_frp
source_member,,,,,,,,
fire_archive_J1V-C2_848899.csv,0,0,0,0,0,0,0,5
fire_archive_J2V-C2_848900.csv,0,0,0,0,0,0,0,16
fire_archive_M-C61_848898.csv,0,0,0,0,0,0,0,9
fire_archive_SV-C2_848901.csv,0,0,0,0,0,0,0,1
fire_nrt_J1V-C2_848899.csv,0,0,0,0,0,44689,0,0
fire_nrt_J2V-C2_848900.csv,0,0,0,0,0,41185,0,3
fire_nrt_M-C61_848898.csv,0,0,0,0,0,4264,0,0
fire_nrt_SV-C2_848901.csv,0,0,0,0,0,43565,0,12


Los vacíos del campo `type` de los NRT son estructurales y se preservan como
desconocidos. Los ceros FRP se registran sin excluirlos del conteo.

In [15]:
display(checks[checks.indicator.str.startswith('candidate_')].pivot(
    index='source_member', columns='indicator', values='true_rows').fillna(0))
profile = pd.read_csv(root / 'valores_categoricos.csv', dtype={'value': 'string'})
display(profile[profile.field.isin(['version', 'type'])])

indicator,candidate_confidence_filter,candidate_pilot_scope,candidate_vegetation_filter
source_member,,,
fire_archive_J1V-C2_848899.csv,601226,0,624127
fire_archive_J2V-C2_848900.csv,227778,0,235493
fire_archive_M-C61_848898.csv,340203,158674,352951
fire_archive_SV-C2_848901.csv,997115,966592,1031180
fire_nrt_J1V-C2_848899.csv,41046,0,0
fire_nrt_J2V-C2_848900.csv,37523,0,0
fire_nrt_M-C61_848898.csv,4046,0,0
fire_nrt_SV-C2_848901.csv,40334,0,0


,source_member,field,value,rows
101,fire_archive_M-C61_848898.csv,version,6.03,312995
102,fire_archive_M-C61_848898.csv,version,61.03,49164
103,fire_archive_M-C61_848898.csv,type,0,352951
104,fire_archive_M-C61_848898.csv,type,2,8418
105,fire_archive_M-C61_848898.csv,type,1,628
106,fire_archive_M-C61_848898.csv,type,3,162
206,fire_nrt_M-C61_848898.csv,version,6.1NRT,4264
215,fire_archive_J1V-C2_848899.csv,version,2,641945
216,fire_archive_J1V-C2_848899.csv,type,0,624127
217,fire_archive_J1V-C2_848899.csv,type,2,12002


## Evaluación y siguientes decisiones
La comparación inicial utilizará Aqua MODIS y Suomi NPP VIIRS, SP 2013–2025.
Se reservarán años completos antes de ajustar modelos. No hay todavía target
calibrado, baseline, modelo ni métricas de error de armonización.
Pendientes: revisar filtros, elegir malla y proyección, definir semana y corte
de validación. No se imputarán ceros en días sin cobertura conocida.

## Limitaciones
Se validaron conteos exportados, confianza, versión, fecha-hora y hashes de origen.
No se comprobó pertenencia exacta al polígono nacional ni cobertura de nubes/pasos.
Los conteos por plataforma son diagnósticos, no una serie armonizada.